<a href="https://colab.research.google.com/github/Mustafizb24/gold-directional-forecasting/blob/main/gold_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
installs yfinance, which is a free library to download market data directly from Yahoo Finance
Also imports pandas, for working with dataframes
'''

!pip install yfinance -q
import yfinance as yf
import pandas as pd



In [2]:
# Downoad daily closing prices for gold futures and the US Dollar Index
yahoo = yf.download(["GC=F", "DX-Y.NYB"], start = "2010-01-01", end = "2026-01-01", auto_adjust=True) ["Close"] #Close means it records the price at the end of each daily session

yahoo.head() #Shows the first 5 rows

[*********************100%***********************]  2 of 2 completed


Ticker,DX-Y.NYB,GC=F
Date,,
2010-01-04,77.529999,1117.699951
2010-01-05,77.620003,1118.099976
2010-01-06,77.489998,1135.900024
2010-01-07,77.910004,1133.099976
2010-01-08,77.470001,1138.199951


In [3]:
# Download the US 10-year Treasury yield from FRED (Federal Reserve)
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10"
fred = pd.read_csv(url, index_col = 0, parse_dates = True, na_values=".")

#Keep only 2010-2025 to match the Yahoo data for gold and dollar index
fred = fred.loc["2010-01-01":"2025-12-31"]

fred.head()

,DGS10
observation_date,
2010-01-01,NaN
2010-01-04,3.85
2010-01-05,3.77
2010-01-06,3.85
2010-01-07,3.85


In [4]:
# Combine the Yahoo and FRED data into one table, using Yahoo's trading days
raw = yahoo.join(fred, how="left")

# Rename the columns to short, clear names
raw = raw.rename(columns={"GC=F": "gold", "DX-Y.NYB": "dxy", "DGS10" : "yield_10y"})

#Save as a CSV
raw.to_csv("raw_2010_2025.csv")

raw.head()

,dxy,gold,yield_10y
Date,,,
2010-01-04,77.529999,1117.699951,3.85
2010-01-05,77.620003,1118.099976,3.77
2010-01-06,77.489998,1135.900024,3.85
2010-01-07,77.910004,1133.099976,3.85
2010-01-08,77.470001,1138.199951,3.83


In [5]:
#Checking number of rows and columns
print(raw.shape)

# Count missing values in each column
print(raw.isna().sum())

#Show last 5 rows
raw.tail()

(4027, 3)
dxy           2
gold          3
yield_10y    32
dtype: int64


,dxy,gold,yield_10y
Date,,,
2025-12-24,97.980003,4502.799805,4.15
2025-12-26,98.019997,4552.700195,4.14
2025-12-29,98.040001,4343.600098,4.12
2025-12-30,98.239998,4386.299805,4.14
2025-12-31,98.279999,4341.100098,4.18


4000 rows lines up with approximately 16 years of trading days

In [6]:
# Removing rows where there is a null value for gold
clean = raw.dropna(subset=["gold"])

# Fill remaining gaps with the previous day's value
clean = clean.ffill()

# Check: there should now be no missing values
print(clean.shape)
print(clean.isna().sum())

(4024, 3)
dxy          0
gold         0
yield_10y    0
dtype: int64


In [7]:
# Target: 1 if tomorrow's gold close is higher than today's otherwise is 0
clean["target"] = (clean["gold"].shift(-1) > clean["gold"]).astype(int) #shift -1 means the column will move up one row to line up with next day's value

# Removing last row: so there's no tomorrow to compare it with
clean = clean.iloc[:-1]

clean.tail()

,dxy,gold,yield_10y,target
Date,,,,
2025-12-23,97.940002,4505.700195,4.18,0
2025-12-24,97.980003,4502.799805,4.15,1
2025-12-26,98.019997,4552.700195,4.14,0
2025-12-29,98.040001,4343.600098,4.12,1
2025-12-30,98.239998,4386.299805,4.14,0


In [8]:
# Count how many up (1) and down (0) days there are
print(clean["target"].value_counts())

print(clean["target"].value_counts(normalize=True) * 100) #As a percentage

target
1    2118
0    1905
Name: count, dtype: int64
target
1    52.647278
0    47.352722
Name: proportion, dtype: float64


For majority class baseline, the classes are 52.6% up and 47.4% down, this means the baseline already achieves a score of 52.6^. Any model needs to clearly beat that, so accuracy alone shouldn't be prioritised

In [9]:
# Saving the cleaned dataset with the target column
clean.to_csv("clean_2010_2025.csv")

In [11]:
import numpy as np

# Daily log returns for gold and the dollar index
clean["gold_ret"]  = np.log(clean["gold"] / clean["gold"].shift(1))
clean["dxy_ret"] = np.log(clean["dxy"] / clean["dxy"].shift(1))

# Using log returns instead of % change because they add up over time, which percentagees don't measure

# Daily change in the 10-year yield (in %)
clean["yield_chg"] = clean ["yield_10y"].diff()

clean.head().round(4)

,dxy,gold,yield_10y,target,gold_ret,dxy_ret,yield_chg
Date,,,,,,,
2010-01-04,77.53,1117.7,3.85,1,NaN,NaN,NaN
2010-01-05,77.62,1118.1,3.77,1,0.0004,0.0012,-0.08
2010-01-06,77.49,1135.9,3.85,0,0.0158,-0.0017,0.08
2010-01-07,77.91,1133.1,3.85,1,-0.0025,0.0054,0.00
2010-01-08,77.47,1138.2,3.83,1,0.0045,-0.0057,-0.02


In [12]:
# Gold's return from 1 and 2 days ago
clean["gold_ret_lag1"] = clean["gold_ret"].shift(1)
clean["gold_ret_lag2"] = clean["gold_ret"].shift(2) #Positive numbers mean it looks at the previous rows up

clean.head()

,dxy,gold,yield_10y,target,gold_ret,dxy_ret,yield_chg,gold_ret_lag1,gold_ret_lag2
Date,,,,,,,,,
2010-01-04,77.529999,1117.699951,3.85,1,NaN,NaN,NaN,NaN,NaN
2010-01-05,77.620003,1118.099976,3.77,1,0.000358,0.001160,-0.08,NaN,NaN
2010-01-06,77.489998,1135.900024,3.85,0,0.015795,-0.001676,0.08,0.000358,NaN
2010-01-07,77.910004,1133.099976,3.85,1,-0.002468,0.005405,0.00,0.015795,0.000358
2010-01-08,77.470001,1138.199951,3.83,1,0.004491,-0.005664,-0.02,-0.002468,0.015795


In [14]:
# 20-day and 50-day moving averages of gold's price. Rolling.mean function gives average over last n days
#20 days for short term trends and 50 days for medium term trends
ma20 = clean["gold"].rolling(20).mean()
ma50 = clean["gold"].rolling(50).mean()

# How far today's price is above(+) or below (-) each average
clean["gold_ma20_gap"] = clean["gold"] / ma20 - 1
clean["gold_ma50_gap"] = clean["gold"] / ma50 - 1

clean[["gold", "gold_ma20_gap", "gold_ma50_gap"]].tail()

,gold,gold_ma20_gap,gold_ma50_gap
Date,,,
2025-12-23,4505.700195,0.049516,0.079237
2025-12-24,4502.799805,0.044428,0.076791
2025-12-26,4552.700195,0.051278,0.086899
2025-12-29,4343.600098,0.001968,0.036786
2025-12-30,4386.299805,0.010518,0.046114


In [15]:
# 20-day volatility: how much gold's daily returns have varied over the last month
clean["gold_vol20"] = clean["gold_ret"].rolling(20).std()

#Standard deviation is a measure of how spread out values are, so here it measures how much gold's daily returns have been fluctuating

clean[["gold_ret", "gold_vol20"]].tail()

,gold_ret,gold_vol20
Date,,
2025-12-23,0.008089,0.008557
2025-12-24,-0.000644,0.008503
2025-12-26,0.011021,0.008631
2025-12-29,-0.047017,0.013662
2025-12-30,0.009782,0.013781


In [16]:
# Remove the warm-up rows (early row where feature can't be calculated) with blanks (from return, lags and rolling windows)
features = clean.dropna()

# Check: size and total numbers of missing values
print(features.shape)
print(features.isna().sum().sum()) #First .sum counts the blanks per column, the second adds those into one total
features.head()

(3974, 12)
0


,dxy,gold,yield_10y,target,gold_ret,dxy_ret,yield_chg,gold_ret_lag1,gold_ret_lag2,gold_ma20_gap,gold_ma50_gap,gold_vol20
Date,,,,,,,,,,,,
2010-03-16,79.750000,1122.199951,3.66,1,0.015355,-0.006250,-0.05,0.003263,-0.005884,0.004129,0.010257,0.008323
2010-03-17,79.639999,1124.000000,3.65,1,0.001603,-0.001380,-0.01,0.015355,0.003263,0.005538,0.011763,0.008329
2010-03-18,80.220001,1127.400024,3.68,0,0.003020,0.007256,0.03,0.001603,0.015355,0.008155,0.014654,0.008344
2010-03-19,80.720001,1107.400024,3.70,0,-0.017899,0.006214,0.02,0.003020,0.001603,-0.009113,-0.002835,0.009263
2010-03-22,80.650002,1099.300049,3.67,1,-0.007341,-0.000868,-0.03,-0.017899,0.003020,-0.015775,-0.009525,0.009245
